# Unsupervised & Reinforcement Learning: Clustering, PCA, and Q-Learning

A compact practical covering two core unsupervised learning techniques (K-Means clustering and PCA) followed by a from-scratch Q-learning agent solving the FrozenLake environment.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

## Part 1: Unsupervised Learning
### 1.1 K-Means Clustering

Generate synthetic blob data and cluster it with K-Means.

In [ ]:
X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.9, random_state=42)

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = kmeans.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X[:, 0], X[:, 1], c=y_true, cmap='viridis', s=20)
axes[0].set_title('Ground Truth')
axes[1].scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=20)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
                c='red', marker='X', s=200, label='Centroids')
axes[1].set_title('K-Means Clustering')
axes[1].legend()
plt.tight_layout()
plt.show()

Choosing `k` with the elbow method (inertia vs. number of clusters).

In [ ]:
inertias = []
k_range = range(1, 10)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertias, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

### 1.2 Dimensionality Reduction with PCA

Project a higher-dimensional dataset down to 2D for visualization.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine()
X_wine = StandardScaler().fit_transform(wine.data)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_wine)

print('Explained variance ratio:', pca.explained_variance_ratio_)
print('Total variance captured:', pca.explained_variance_ratio_.sum())

plt.figure(figsize=(6, 5))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=wine.target, cmap='viridis', s=30)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.title('Wine Dataset Projected via PCA')
plt.colorbar(scatter, label='Class')
plt.show()

## Part 2: Reinforcement Learning
### Q-Learning on FrozenLake

A tabular Q-learning agent that learns a policy to cross a slippery frozen lake without falling into a hole, using the classic Bellman update.

In [ ]:
import gymnasium as gym

env = gym.make('FrozenLake-v1', is_slippery=True)
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

alpha = 0.1        # learning rate
gamma = 0.99       # discount factor
epsilon = 1.0      # exploration rate
epsilon_min = 0.01
epsilon_decay = 0.9995
n_episodes = 10000

rewards_per_episode = []

for episode in range(n_episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()
        else:
            action = np.argmax(Q[state])

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        Q[state, action] += alpha * (
            reward + gamma * np.max(Q[next_state]) - Q[state, action]
        )

        state = next_state
        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_per_episode.append(total_reward)

env.close()

Plot the rolling success rate to see the agent learn over time.

In [ ]:
window = 100
rolling_success = np.convolve(rewards_per_episode, np.ones(window) / window, mode='valid')

plt.figure(figsize=(8, 4))
plt.plot(rolling_success)
plt.xlabel('Episode')
plt.ylabel(f'Success rate (rolling mean over {window} episodes)')
plt.title('Q-Learning on FrozenLake: Learning Curve')
plt.show()

print(f'Final rolling success rate: {rolling_success[-1]:.2%}')

Visualize the greedy policy learned by the agent (arrows show the best action in each state).

In [ ]:
action_arrows = {0: '←', 1: '↓', 2: '→', 3: '↑'}  # Left, Down, Right, Up
grid_size = int(np.sqrt(n_states))
policy = np.argmax(Q, axis=1)

policy_grid = [action_arrows[policy[s]] for s in range(n_states)]

for r in range(grid_size):
    print(' '.join(policy_grid[r * grid_size:(r + 1) * grid_size]))

## Summary

- **K-Means** partitioned unlabeled data into meaningful clusters, and the elbow method helped select a reasonable `k`.
- **PCA** compressed a higher-dimensional dataset into two components while retaining most of the variance, making it easy to visualize class separation.
- **Q-learning** learned an optimal policy purely from trial-and-error interaction with the environment, with no labeled data or expert supervision.